# Clase 06: Resolución de problemas usando estructuras de datos

## Josephus Problem I

Problema original: [CSES Problem Set - Josephus Problem I](https://cses.fi/problemset/task/2162)

Hoy no empezamos con una estructura de datos.

Empezamos con un problema.

## Objetivos

Al terminar este notebook deberás poder:

- leer estratégicamente un problema;
- identificar entrada y salida;
- reconocer restricciones;
- decidir qué información debe mantenerse;
- elegir una estructura de datos;
- escribir pseudocódigo;
- implementar una solución;
- ejecutar pruebas públicas;
- escribir una discusión técnica.

## Cambio de enfoque

Antes seguíamos esta ruta:

```text
Estructura -> Implementación -> Pruebas
```

A partir de hoy seguiremos esta otra:

```text
Problema -> Lectura estratégica -> Modelado -> Elección de estructura
         -> Pseudocódigo -> Implementación -> Pruebas -> Discusión técnica
```

Las estructuras de datos son herramientas para resolver problemas, no una lista de definiciones para memorizar.

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

raiz = Path.cwd()
if not (raiz / "src").exists() and (raiz.parent / "src").exists():
    raiz = raiz.parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

print(f"Carpeta de trabajo detectada: {raiz}")

## 1. Lectura estratégica

El problema describe un juego con `n` niños numerados del `1` al `n` en un círculo.

Durante el juego se elimina uno de cada dos niños hasta que no queda nadie.

La salida debe ser el orden en que los niños son eliminados.

Ejemplo oficial:

```text
Entrada:
7

Salida:
2 4 6 1 5 3 7
```

Antes de programar, necesitamos entender exactamente qué se mueve, qué se conserva y qué se imprime.

### Entrada, salida y restricciones

Completa en tu copia del notebook:

```text
Entrada:
- TODO

Salida:
- TODO

Restricción principal:
- TODO
```

Dato importante del enunciado: `1 <= n <= 2 * 10^5`.

Eso significa que una solución demasiado lenta puede fallar aunque sea correcta en ejemplos pequeños.

## ¿Por qué este problema?

Josephus Problem I es una buena primera clase de resolución porque:

- modela naturalmente una simulación;
- una cola aparece casi inevitablemente;
- obliga a razonar antes de programar;
- no requiere grafos;
- no requiere árboles;
- conecta con FIFO;
- permite discutir complejidad;
- tiene pruebas públicas pequeñas pero también restricciones grandes.

# Ingeniería inversa del algoritmo

Esta sección será permanente en las siguientes clases.

Antes de escribir una sola línea de código, responde:

1. ¿Qué está pidiendo exactamente el problema?
2. ¿Qué información debo recordar mientras avanzo?
3. ¿Qué operaciones realizo continuamente?
4. ¿Existe una estructura de datos que implemente naturalmente esas operaciones?
5. ¿Cómo resolvería este problema con papel y lápiz?
6. Escribir únicamente después el pseudocódigo.

Estas respuestas deben aparecer en `notebook.md`, en el mismo orden del notebook. `notebook.md` es un documento de trabajo: no debe contener código completo.

## Entregable `notebook.md`

Al terminar la clase, crea un archivo llamado `notebook.md`.

Este archivo no es copia del notebook. Contiene únicamente tus respuestas argumentadas a las preguntas del notebook, en el mismo orden.

Estructura sugerida:

```text
# Notebook - Clase 06

## Lectura estratégica
### ¿Qué está pidiendo el problema?
...
### ¿Qué información debemos recordar?
...

---

## Ingeniería inversa del algoritmo
...

---

## Modelado
...

---

## Pseudocódigo
...
```

## 2. Modelado

Pensemos en `n = 7`.

```text
1  2  3  4  5  6  7
```

No basta saber cuáles números existen.

Necesitamos mantener:

- quién sigue vivo;
- en qué orden circular están;
- quién se salva en el turno actual;
- quién se elimina;
- el orden de eliminaciones.

### Preguntas de modelado

Responde antes de seguir:

1. Cuando un niño se salva, ¿desaparece del problema?
2. Cuando un niño se elimina, ¿vuelve a participar?
3. ¿Qué significa avanzar en un círculo si estamos usando una estructura lineal?
4. ¿Qué operación se repite una y otra vez?

In [ ]:
# Simulación de un solo paso con n = 7.
# No es todavía la solución completa; solo modela un turno.

vivos = deque([1, 2, 3, 4, 5, 6, 7])

salvado = vivos.popleft()
vivos.append(salvado)
eliminado = vivos.popleft()

print("Se salva:", salvado)
print("Se elimina:", eliminado)
print("Vivos en orden circular:", list(vivos))

assert eliminado == 2

## 3. Descubrir la cola

Observa el paso anterior:

1. Tomamos al niño del frente.
2. Si se salva, lo mandamos al final.
3. Tomamos al siguiente del frente.
4. Si se elimina, lo guardamos en la respuesta.

Eso suena a una cola:

```text
frente -> sale -> quizá regresa al final
```

La estructura no apareció por nombre. Apareció por las operaciones que el problema exige.

### Comparación breve con pila

Una pila responde a LIFO: sale el último que entró.

Este problema no necesita eso.

Aquí necesitamos conservar el orden circular y atender desde el frente. Por eso una pila no representa naturalmente el proceso.

In [ ]:
# Pequeño experimento: tres turnos manuales.
# La función solo se usa para observar, no para entregar una solución final.

def mostrar_turnos_iniciales(n, pasos):
    vivos = deque(range(1, n + 1))
    eliminados = []
    historia = []

    for _ in range(pasos):
        salvado = vivos.popleft()
        vivos.append(salvado)
        eliminado = vivos.popleft()
        eliminados.append(eliminado)
        historia.append((salvado, eliminado, list(vivos)))

    return eliminados, historia

eliminados, historia = mostrar_turnos_iniciales(7, 3)
print("Primeras eliminaciones:", eliminados)
for i, (salvado, eliminado, estado) in enumerate(historia, start=1):
    print(f"Turno {i}: se salva {salvado}, se elimina {eliminado}, quedan {estado}")

## 4. Diseñar el algoritmo en español

Antes del pseudocódigo, escríbelo en lenguaje natural.

Una versión posible es:

```text
Crear una cola con los niños del 1 al n.
Mientras haya niños vivos:
    mover al final al niño que se salva
    eliminar al siguiente niño
    guardar el eliminado en la respuesta
Regresar la respuesta
```

Ahora explica con tus palabras por qué esto respeta la regla de eliminar uno de cada dos.

## Pseudocódigo

Completa los espacios marcados:

```text
función orden_eliminacion(n):
    validar n
    vivos = cola con números de 1 a n
    eliminados = lista vacía

    mientras vivos no esté vacía:
        si solo queda un niño:
            eliminarlo y guardarlo
        si no:
            TODO: mover al final al niño que se salva
            TODO: eliminar al siguiente niño
            TODO: guardar el eliminado

    regresar eliminados
```

Hay varias formas de escribirlo. Lo importante es que cada operación tenga sentido en el modelo.

## 5. Implementación

El archivo base está en:

```text
src/josephus.py
```

La función central está marcada como TODO:

```python
def orden_eliminacion(n: int) -> list[int]:
    ...
```

Tu entrega final de código debe ir en `entregas/clase_06/nombre/implementacion.py`.

Tus respuestas guiadas del notebook deben ir en `entregas/clase_06/nombre/notebook.md`.

In [ ]:
from src.josephus import orden_eliminacion, formatear_salida

try:
    print(orden_eliminacion(7))
except NotImplementedError as error:
    print("Pendiente esperado:", error)

### Espacio de implementación guiada

Puedes usar esta celda para practicar, pero tu entrega final debe estar en `implementacion.py`.

Mantén esta celda ejecutable aunque todavía no tengas la solución completa.

In [ ]:
def orden_eliminacion_borrador(n):
    """Borrador de trabajo. Completa los TODO en tu copia."""
    if n < 1:
        raise ValueError("n debe ser al menos 1")

    vivos = deque(range(1, n + 1))
    eliminados = []

    # TODO alumno: reemplaza este borrador por tu simulación.
    # Pista: piensa en una operación para quien se salva y otra para quien sale.

    return eliminados

print("Borrador para n=7:", orden_eliminacion_borrador(7))

## Pruebas pequeñas con `assert`

Antes de `pytest`, conviene escribir expectativas pequeñas.

Completa estas pruebas cuando tengas tu implementación:

```python
assert orden_eliminacion(1) == [1]
assert orden_eliminacion(2) == [2, 1]
assert orden_eliminacion(7) == [2, 4, 6, 1, 5, 3, 7]
```

Estas pruebas verifican casos conocidos, pero no garantizan que todos los casos funcionen.

In [ ]:
# Esta celda no falla mientras la función esté pendiente.
# Cuando implementes orden_eliminacion, cambia EJECUTAR_ASSERTS a True.

EJECUTAR_ASSERTS = False

if EJECUTAR_ASSERTS:
    assert orden_eliminacion(1) == [1]
    assert orden_eliminacion(2) == [2, 1]
    assert orden_eliminacion(7) == [2, 4, 6, 1, 5, 3, 7]
    print("Asserts básicos superados")
else:
    print("Asserts desactivados hasta completar la implementación")

## 6. Pruebas públicas y privadas

Las pruebas públicas están en:

```text
tests/test_publico_josephus.py
```

Sirven como retroalimentación.

Las pruebas privadas no están en este repositorio. Pueden incluir:

- casos límite;
- entradas grandes;
- errores comunes;
- validaciones adicionales.

Que una prueba pública pase no significa que la solución esté completamente probada.

In [ ]:
# Revisemos qué pruebas públicas existen sin ejecutarlas todavía.
pruebas = raiz / "tests" / "test_publico_josephus.py"
print(pruebas.read_text(encoding="utf-8")[:2000])

### Ejecutar pytest

Cuando tengas una implementación, ejecuta en terminal:

```bash
pytest -v
```

Observación.

En algunos sistemas operativos o configuraciones de Python, el comando `pytest` puede no encontrar correctamente el entorno del proyecto.

Si esto ocurre, utiliza:

```bash
python3 -m pytest -v
```

Este comando ejecuta `pytest` utilizando explícitamente el intérprete de Python y suele resolver problemas relacionados con múltiples instalaciones de Python o con el `PATH`.

En tu entrega debes crear `reporte_pytest.md`. El reporte debe usar la salida obtenida con `pytest -v` o, si fue necesario, `python3 -m pytest -v`.

Debe incluir: comando utilizado, salida completa, interpretación, número de pruebas, cuántas pasaron, qué comportamiento verifican y qué caso importante todavía no está siendo probado.

In [ ]:
# Esta celda solo recolecta pruebas para verificar que pytest puede encontrarlas.
# No ejecuta las pruebas completas porque la solución todavía está pendiente.

EJECUTAR_COLECCION = False

if EJECUTAR_COLECCION:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "--collect-only", "-q"],
        cwd=raiz,
        text=True,
        capture_output=True,
    )
    print(resultado.stdout)
    print(resultado.stderr)
else:
    print("Colección de pytest desactivada en el notebook guiado")

## 7. Discusión técnica

Tu entrega debe incluir `notebook.md` y `discusion.md`.

`notebook.md` es un documento de trabajo: contiene tus respuestas guiadas del notebook, en el mismo orden, sin código completo.

`discusion.md` es un documento técnico: argumenta decisiones de diseño, pruebas, complejidad y contrastes. No debe repetir literalmente `notebook.md`.

`discusion.md` debe contener estas secciones:

1. Lectura estratégica.
2. Elección de estructura.
3. Diseño del algoritmo.
4. Pruebas.
5. Complejidad.
6. Contraste.
7. Pregunta abierta.

No basta decir: "usé una cola porque sí".

Debes argumentar qué operaciones del problema hacen natural esa elección.

## Complejidad

Discute:

- ¿Cuántos niños se eliminan?
- ¿Cuántas veces se mueve o procesa cada niño?
- ¿Qué estructura permite que esas operaciones sean eficientes?
- ¿Cuánta memoria adicional se usa?

Escribe tu análisis con notación Big-O y con una explicación en palabras.

## Problema secundario: Nearest Smaller Values

Problema original: [CSES Problem Set - Nearest Smaller Values](https://cses.fi/problemset/task/1645)

Solo lo usaremos como motivación.

La tarea es: para cada posición de un arreglo, encontrar la posición más cercana a la izquierda que tenga un valor menor.

Ejemplo:

```text
2 5 1 4 8 3 2 5
0 1 0 3 4 3 3 7
```

No lo resolveremos hoy.

### Discusión: ¿por qué una cola deja de funcionar aquí?

Contrasta con Josephus:

- En Josephus, el orden de atención está determinado por el círculo.
- En Nearest Smaller Values, necesitamos recordar candidatos de la izquierda.
- Algunos candidatos dejan de servir cuando aparece un valor más pequeño.
- No basta atender en orden FIFO.

Pregunta para la siguiente clase:

¿Qué estructura permite conservar candidatos útiles y descartar candidatos inútiles?

## Entrega

No entregues este archivo `.ipynb`.

Entrega:

```text
entregas/
└── clase_06/
    └── nombre/
        ├── implementacion.py
        ├── notebook.md
        ├── discusion.md
        └── reporte_pytest.md
```

`implementacion.py` debe ser importable por `pytest` y no debe ejecutar `input()` al importarse.

`notebook.md` contiene respuestas guiadas del notebook.

`discusion.md` contiene argumentación técnica y no debe copiar literalmente `notebook.md`.

## Cierre

Preguntas de contraste para discutir:

1. Si se eliminara cada tercer niño, ¿la cola seguiría siendo natural?
2. Si necesitáramos consultar siempre el menor número vivo, ¿la cola bastaría?
3. Si el círculo cambiara dinámicamente con inserciones en medio, ¿qué parte del modelo se rompería?
4. ¿Qué prueba pública te da más confianza y cuál falta?